# Interaction Map — simulate, analyse, design

Give this notebook an IDR sequence. It simulates the ensemble, measures which
residue pairs interact more or less than an interaction-free chain would, draws
the Interaction Map, and ranks positions worth mutating.

**Simulation** is the CALVADOS2 coarse-grained engine from IDRLab, unchanged.
CALVADOS and the IDRLab notebook are from the Lindorff-Larsen group:
https://github.com/KULL-Centre/CALVADOS

**Analysis** is the `interaction_map` package, installed from GitHub rather than
copied in, so this notebook cannot drift away from the tested code.

> Runtime → Change runtime type → **GPU**. The simulation needs CUDA.


## 1 — Your sequence

Run this cell first. It records the settings before the environment setup restarts
the kernel.


In [ ]:
#@title <b><font color='#E3B505'>1 - Input protein sequence</font></b>
import numpy as np
import os
import shutil
import ipywidgets as widgets

try:
    os.rmdir('sample_data')
except:
    pass
#@markdown Name the IDR that you want to simulate:
NAME = "Q99457_1_97" #@param {type:"string"}

#@markdown Insert here the sequence of the IDR that you want to simulate:
SEQUENCE = "MAEADFKMVSEPVAHGVAEEEMASSTSDSGEESDSSSSSSSTSDSSSSSSTSGSSSGSGSSSSSSGSTSSRSRLYRKKRVPEPSRRARRAPLGTNFV" #@param {type:"string"}
if " " in SEQUENCE:
    SEQUENCE = ''.join(SEQUENCE.split())
    print('Blank character(s) found in the provided sequence. Sequence has been corrected, but check for integrity:')
    print(SEQUENCE)
    print('\n')
#@markdown Simulation settings:
Temperature = 310 #@param {type:"number"}
Ionic_strength = 0.150 #@param {type:"number"}
#@markdown <i>*Units: Temperature [K], Ionic_strength [M]<i>

#@markdown Define charge states:
charged_N_terminal_amine = True #@param {type:"boolean"}
charged_C_terminal_carboxyl = False #@param {type:"boolean"}
charged_histidine = False #@param {type:"boolean"}
Nc = 1 if charged_N_terminal_amine == True else 0
Cc = 1 if charged_C_terminal_carboxyl == True else 0

if charged_histidine == True:
    print('Define pH and pKa to set the charge of Histidines according to the Henderson-Hasselbalch equation.')
    pH = input('Enter pH value: ')
    pH = float(pH)
    pKa = input('Enter pKa value: ')
    pKa = float(pKa)
    Hc = 1/(1+10**(pH-pKa))
if charged_histidine == False:
    Hc = 0

np.savetxt('env_settings.txt', np.array([Temperature, Ionic_strength, Hc, Nc, Cc]).T, header='temperature ionic_strength, His_charge, N_term_charge, C_term_charge')

# Need to store metadata prior to condacolab restarting the kernel
f = open('seq.fasta','w')
f.write('>{:s}\n{:s}'.format(NAME,SEQUENCE))
f.close()

## 2 — Environment

Two cells, a few minutes. The kernel restarts after the first; that is expected.


In [ ]:
#@title <b>Preliminary operations</b>: setting the environment (i)
import subprocess
subprocess.run( 'pip install -q condacolab'.split() )
import condacolab
condacolab.install()

In [ ]:
#@title <b>Preliminary operations</b>: setting the environment (ii)
import subprocess
REPO = 'https://github.com/smallfishabc/interaction_map.git'
BRANCH = 'refactor_20261024'
print('Installing libraries...')
# mdtraj is pinned. 1.10 and later change behaviour this analysis depends on.
_ = subprocess.run( 'mamba install matplotlib mdtraj=1.9.9 openmm=8.2 -c conda-forge --yes'.split() )
subprocess.run( 'pip install wget localcider==0.1.18'.split() )
# The interaction map analysis, installed rather than pasted in, so this notebook
# and the repository cannot drift apart. --no-deps because conda supplied them.
subprocess.run( ('pip install --no-deps git+' + REPO + '@' + BRANCH).split() )
import wget
wget.download('https://raw.githubusercontent.com/fpesceKU/BLOCKING/main/block_tools.py')
wget.download('https://raw.githubusercontent.com/fpesceKU/BLOCKING/main/main.py')
import os
import shutil
import itertools
import numpy as np
import pandas as pd
import scipy.stats as scs
from scipy.optimize import curve_fit
from sklearn.covariance import LedoitWolf
from scipy import constants
from numpy import linalg
from localcider.sequenceParameters import SequenceParameters
from main import BlockAnalysis
import mdtraj as md
import openmm
from openmm import unit, app
import matplotlib as mpl
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable
from google.colab import files
import matplotlib_inline.backend_inline
matplotlib_inline.backend_inline.set_matplotlib_formats('pdf', 'svg')
from interaction_map import normalization as nl, interaction_plot as ip, profiles
from interaction_map import contact_map_generation as cg
from interaction_map.util import mutation_scan as scan
print('mdtraj', md.version.full_version)


## 3 — Simulation engine

The CALVADOS2 force field and the OpenMM setup, from IDRLab unchanged.


In [ ]:
#@title <b><font color='#A79AB2'>MD Toolbox</font></b>
url = 'https://github.com/KULL-Centre/_2023_Tesei_IDRome/blob/main'
wget.download(url+'/md_simulations/data/residues.csv?raw=true')

residues = pd.read_csv('residues.csv')
residues = residues.set_index('one')

def genParamsLJ(df,seq,Nc,Cc):
    fasta = seq.copy()
    r = df.copy()
    if Nc == 1:
        r.loc['X'] = r.loc[fasta[0]]
        r.loc['X','MW'] += 2
    if Cc == 1:
        r.loc['Z'] = r.loc[fasta[-1]]
        r.loc['Z','MW'] += 16
    lj_eps = 0.2*4.184
    lj_sigma = pd.DataFrame((r.sigmas.values+r.sigmas.values.reshape(-1,1))/2,
                            index=r.sigmas.index,columns=r.sigmas.index)
    lj_lambda = pd.DataFrame((r.lambdas.values+r.lambdas.values.reshape(-1,1))/2,
                             index=r.lambdas.index,columns=r.lambdas.index)
    return lj_eps, lj_sigma, lj_lambda

def genParamsDH(df,seq,temp,ionic,Nc,Cc,Hc):
    kT = 8.3145*temp*1e-3
    fasta = seq.copy()
    r = df.copy()
    # Set the charge on HIS based on the pH of the protein solution
    r.loc['H','q'] = Hc
    if Nc == 1:
        r.loc['X'] = r.loc[fasta[0]]
        r.loc['X','q'] = r.loc[seq[0],'q'] + 1.
        fasta[0] = 'X'
    if Cc == 1:
        r.loc['Z'] = r.loc[fasta[-1]]
        r.loc['Z','q'] = r.loc[seq[-1],'q'] - 1.
        fasta[-1] = 'Z'
    # Calculate the prefactor for the Yukawa potential
    fepsw = lambda T : 5321/T+233.76-0.9297*T+0.1417*1e-2*T*T-0.8292*1e-6*T**3
    epsw = fepsw(temp)
    lB = 1.6021766**2/(4*np.pi*8.854188*epsw)*6.022*1000/kT
    yukawa_eps = [r.loc[a].q*np.sqrt(lB*kT) for a in fasta]
    # Calculate the inverse of the Debye length
    yukawa_kappa = np.sqrt(8*np.pi*lB*ionic*6.022/10)
    return yukawa_eps, yukawa_kappa

def genDCD(name, seq, eqsteps=10):
    """
    Generates coordinate and trajectory
    in convenient formats
    """
    traj = md.load("{:s}/pretraj.dcd".format(name), top="{:s}/top.pdb".format(name))
    cgtop = md.Topology()
    cgchain = cgtop.add_chain()
    for aa in seq:
        cgres = cgtop.add_residue(aa, cgchain)
        cgtop.add_atom('CA', element=md.element.carbon, residue=cgres)
    for i in range(traj.n_atoms-1):
        cgtop.add_bond(cgtop.atom(i),cgtop.atom(i+1))
    traj = md.Trajectory(traj.xyz, cgtop, traj.time, traj.unitcell_lengths, traj.unitcell_angles)
    traj = traj.image_molecules(inplace=False, anchor_molecules=[set(traj.top.chain(0).atoms)], make_whole=True)
    traj.center_coordinates()
    traj.xyz += traj.unitcell_lengths[0,0]/2
    traj[int(eqsteps):].save_dcd("{:s}/traj.dcd".format(name))
    traj[int(eqsteps)].save_pdb("{:s}/top.pdb".format(name))

def simulate(residues,name,seq,temp,ionic,Nc,Cc,Hc,nsteps,stride=1e3,eqsteps=1000):
    os.mkdir(name)

    lj_eps, _, _ = genParamsLJ(residues,seq,Nc,Cc)
    yukawa_eps, yukawa_kappa = genParamsDH(residues,seq,temp,ionic,Nc,Cc,Hc)

    N = len(seq)
    L = (N-1)*0.38+4

    system = openmm.System()

    # set box vectors
    a = unit.Quantity(np.zeros([3]), unit.nanometers)
    a[0] = L * unit.nanometers
    b = unit.Quantity(np.zeros([3]), unit.nanometers)
    b[1] = L * unit.nanometers
    c = unit.Quantity(np.zeros([3]), unit.nanometers)
    c[2] = L * unit.nanometers
    system.setDefaultPeriodicBoxVectors(a, b, c)

    top = md.Topology()
    pos = []
    chain = top.add_chain()
    pos.append([[0,0,L/2+(i-N/2.)*.38] for i in range(N)])
    for resname in seq:
        residue = top.add_residue(resname, chain)
        top.add_atom(resname, element=md.element.carbon, residue=residue)
    for i in range(chain.n_atoms-1):
        top.add_bond(chain.atom(i),chain.atom(i+1))
    md.Trajectory(np.array(pos).reshape(N,3), top, 0, [L,L,L], [90,90,90]).save_pdb('{:s}/top.pdb'.format(name))

    pdb = app.pdbfile.PDBFile('{:s}/top.pdb'.format(name))

    system.addParticle((residues.loc[seq[0]].MW+2)*unit.amu)
    for a in seq[1:-1]:
        system.addParticle(residues.loc[a].MW*unit.amu)
    system.addParticle((residues.loc[seq[-1]].MW+16)*unit.amu)

    hb = openmm.openmm.HarmonicBondForce()
    energy_expression = 'select(step(r-2^(1/6)*s),4*eps*l*((s/r)^12-(s/r)^6-shift),4*eps*((s/r)^12-(s/r)^6-l*shift)+eps*(1-l))'
    ah = openmm.openmm.CustomNonbondedForce(energy_expression+'; s=0.5*(s1+s2); l=0.5*(l1+l2); shift=(0.5*(s1+s2)/2)^12-(0.5*(s1+s2)/2)^6')
    yu = openmm.openmm.CustomNonbondedForce('q*(exp(-kappa*r)/r - exp(-kappa*4)/4); q=q1*q2')
    yu.addGlobalParameter('kappa',yukawa_kappa/unit.nanometer)
    yu.addPerParticleParameter('q')

    ah.addGlobalParameter('eps',lj_eps*unit.kilojoules_per_mole)
    ah.addPerParticleParameter('s')
    ah.addPerParticleParameter('l')

    for a,e in zip(seq,yukawa_eps):
        yu.addParticle([e*unit.nanometer*unit.kilojoules_per_mole])
        ah.addParticle([residues.loc[a].sigmas*unit.nanometer, residues.loc[a].lambdas*unit.dimensionless])

    for i in range(N-1):
        hb.addBond(i, i+1, 0.38*unit.nanometer, 8033*unit.kilojoules_per_mole/(unit.nanometer**2))
        yu.addExclusion(i, i+1)
        ah.addExclusion(i, i+1)

    yu.setForceGroup(0)
    ah.setForceGroup(1)
    yu.setNonbondedMethod(openmm.openmm.CustomNonbondedForce.CutoffPeriodic)
    ah.setNonbondedMethod(openmm.openmm.CustomNonbondedForce.CutoffPeriodic)
    hb.setUsesPeriodicBoundaryConditions(True)
    yu.setCutoffDistance(4*unit.nanometer)
    ah.setCutoffDistance(2*unit.nanometer)

    system.addForce(hb)
    system.addForce(yu)
    system.addForce(ah)

    #serialized_system = XmlSerializer.serialize(system)
    #outfile = open('system.xml','w')
    #outfile.write(serialized_system)
    #outfile.close()

    integrator = openmm.openmm.LangevinIntegrator(temp*unit.kelvin,0.01/unit.picosecond,0.010*unit.picosecond) #10 fs timestep

    platform = openmm.Platform.getPlatformByName('CUDA')

    simulation = app.simulation.Simulation(pdb.topology, system, integrator, platform, dict(CudaPrecision='mixed'))

    check_point = '{:s}/restart.chk'.format(name)

    if os.path.isfile(check_point):
        print('Reading check point file')
        simulation.loadCheckpoint(check_point)
        simulation.reporters.append(app.dcdreporter.DCDReporter('{:s}/pretraj.dcd'.format(name),int(stride),append=True))
    else:
        simulation.context.setPositions(pdb.positions)
        simulation.minimizeEnergy()
        simulation.reporters.append(app.dcdreporter.DCDReporter('{:s}/pretraj.dcd'.format(name),int(stride)))

    simulation.reporters.append(app.statedatareporter.StateDataReporter('{:s}/traj.log'.format(name),int(stride),
             potentialEnergy=True,temperature=True,step=True,speed=True,elapsedTime=True,separator='\t'))

    simulation.step(nsteps)

    simulation.saveCheckpoint(check_point)

    genDCD(name,[residues.three[aa] for aa in seq],eqsteps)

## 4 — Run the simulation

AUTO picks a length from the sequence. A 60-residue IDR takes a few minutes on a
GPU; longer sequences take proportionally longer.


In [ ]:
#@title <b><font color='#45B69C'>3.1 - Run MD simulation</font></b>
#@markdown Simulation time (ns):
Simulation_time = "AUTO" #@param {type:"raw"}

N_res = len(SEQUENCE)
N_save = 7000 if N_res < 150 else int(np.ceil(3e-4*N_res**2)*1000)

if Simulation_time == "AUTO":
    nsteps = 1010*N_save
    print('AUTO simulation length selected. Running for {} ns'.format(nsteps*0.01/1000))
else:
    nsteps = float(Simulation_time)*1000/0.01//N_save*N_save
try:
    shutil.rmtree(NAME)
except:
    pass
simulate(residues,NAME,list(SEQUENCE),temp=Temperature,ionic=Ionic_strength,Nc=Nc,Cc=Cc,
         Hc=Hc,nsteps=nsteps,stride=N_save,eqsteps=10)

## 5 — The Interaction Map

Contact probability per residue pair, divided by what a glycine–serine linker gives
at the same sequence separation, and logged:

$$I = \ln \frac{p_\mathrm{obs}}{p_\mathrm{ref}}, \qquad p_\mathrm{ref}(s) = a\,s^{\,b}$$

Positive means the pair touches more than an interaction-free chain would.
The `cg` profile carries the constants calibrated for CALVADOS.


In [ ]:
import os
import pandas as pd

profile = profiles.get_profile('cg')
print('cutoff {0} nm, reference {1} * s ** {2}'.format(
    profile.cutoff, profile.reference_a, profile.reference_b))

contact = cg.generate_contact(NAME,
                              os.path.join(NAME, 'top.pdb'),
                              os.path.join(NAME, 'traj.dcd'),
                              profile=profile,
                              sequence=SEQUENCE)
interaction = nl.normalization(contact.contact, profile=profile)
interaction.to_csv(os.path.join(NAME, 'interaction_map.csv'))

levels = interaction['plot_value'].value_counts().to_dict()
print('{0} residue pairs'.format(len(interaction)))
print('strongly attractive {0}, attractive {1}, neutral {2}, repulsive {3}, strongly repulsive {4}'.format(
    levels.get(2, 0), levels.get(1, 0), levels.get(0, 0), levels.get(-1, 0), levels.get(-2, 0)))


### Draw it

Attractive arcs above the sequence, repulsive below. Thickness follows the contact
probability and the strength. Residues are coloured by type: red negative, blue
positive, yellow aromatic, orange hydrophobic, black polar.


In [ ]:
fig, ax = ip.interaction_map(SEQUENCE, len(SEQUENCE), interaction, NAME,
                            profile=profile, show=True, save=True)


## 6 — Which positions to mutate

A strong interaction makes its neighbours look connected too, so reading single
pairs off the map points at a cluster rather than a residue. Each pair is scored
together with its neighbours, with the neighbours counting for less:

$$I_\mathrm{chunk} = \sum_{i}\sum_{j} \frac{I_{ij}}{2^{\,|i-r_1| + |j-r_2|}}$$

Weak and short-range pairs are dropped first, and so are pairs involving the chain
ends, whose interactions are one-sided.


In [ ]:
#@title Filtering
#@markdown Pairs seen less often than this are ignored:
contact_probability_cutoff = 0.01 #@param {type:"slider", min:0, max:0.1, step:0.001}
#@markdown Pairs closer than this along the sequence are ignored:
minimum_separation = 5 #@param {type:"slider", min:1, max:20, step:1}

selected = scan.select_pairs(interaction,
                             contact_probability_cutoff=contact_probability_cutoff,
                             min_separation=minimum_separation,
                             sequence_length=len(SEQUENCE))
scored = scan.add_chunk_strength(selected, SEQUENCE)
print('{0} pairs pass the filter'.format(len(scored)))
scored[['r_1', 'r_1_chunk', 'r_2', 'r_2_chunk', 'distance',
        'cont_prob', 'relative_strength', 'chunk_strength']].head(10)


### Choose what to change

An attractive contact between aromatics is broken by putting a charge there. A
repulsive contact between like charges is relieved by removing one. Pick the kind
of interaction you want to disturb and the kind of residue it involves.


In [ ]:
#@title Target
interaction_type = "attractive" #@param ["attractive", "repulsive"]
residue_types = "aromatic" #@param ["aromatic", "hydrophobic", "charge", "hydrophilic"]
#@markdown Kyte-Doolittle boundary between a hydrophobic and a hydrophilic chunk
#@markdown (the default is glycine). Only used for the hydrophobic and hydrophilic options:
hydrophobicity_cutoff = -1.2 #@param {type:"number"}
#@markdown How many mutants to propose:
number_of_mutants = 10 #@param {type:"slider", min:1, max:50, step:1}

candidates = scan.filter_candidates(scored, interaction_type, residue_types,
                                    hydrophobicity_cutoff)
print('{0} candidate pairs'.format(len(candidates)))
candidates[['r_1', 'r_1_chunk', 'r_2', 'r_2_chunk', 'distance',
            'relative_strength', 'chunk_strength']].head(10)


In [ ]:
mutants = scan.design_mutants(candidates, SEQUENCE, name=NAME,
                              interaction_type=interaction_type,
                              residue_types=residue_types,
                              limit=number_of_mutants)
mutants.to_csv(os.path.join(NAME, 'mutation_candidates.csv'), index=False)
print('{0} mutants proposed'.format(len(mutants)))
mutants


These are candidates, not predictions. The usual next step is to simulate them —
rerun this notebook with one of the sequences above — and compare the ensembles.

Read the ranking before trusting the list. A `chunk_strength` well above the rest
means one region dominates; a flat ranking means the map has no strong preference
and the top of the list is not meaningfully better than the middle.


## 7 — Download


In [ ]:
import shutil
from google.colab import files

archive = shutil.make_archive(NAME, 'zip', NAME)
print('contains: interaction_map.csv, mutation_candidates.csv, the figure,')
print('          the trajectory and the topology')
files.download(archive)
